# MP2 Part 1 — retrieve commits (netid `hvo5`)

Retrieves every commit World of Code has for the 10 projects assigned to `hvo5` in `net2prj.csv`, then records GitHub stars, forks, and the latest commit date.

World of Code (worldofcode.org) is a research archive of open-source version-control data, queried here through its HTTP API. It is not a live mirror of GitHub. The assignment says WoC data ends around January 2025, but the version served by this API includes commits into 2026. Its data still lags behind GitHub, so GitHub shows later commits for most projects. WoC project histories also include commits that are not on the GitHub default branch.

Commit content is requested in chunks of 10 (the API batch size, and under the 50-commit cap) with a 1 second wait between chunks. Results are cached in `df_commits.csv` and `df_commit_data.csv`, so a rerun continues where the last one stopped.


In [ ]:
%pip install -q -U python-woc pandas matplotlib requests tqdm
# please clear the output of this cell in your notebook before checking it in


In [2]:
from pathlib import Path
import csv
import os
import re
import subprocess
import time

import pandas as pd
import requests
from woc.remote import WocMapsRemote

assert Path("net2prj.csv").exists(), "Run this notebook from the folder that contains net2prj.csv."

NETID = "hvo5"
CHUNK = 10
SLEEP_SECONDS = 1
WOC_URL = "https://worldofcode.org/api/"

woc = WocMapsRemote(base_url=WOC_URL)
print("WoC client ready")


WoC client ready


## Assigned projects

The project list comes from `net2prj.csv` (rows for netid `hvo5`). It is not hardcoded.


In [3]:
assigned = pd.read_csv("net2prj.csv")
assigned = assigned[assigned["netID"] == NETID].reset_index(drop=True)
print(f"{len(assigned)} projects assigned to {NETID}")
assigned


10 projects assigned to hvo5


,netID,WoC,GH
0,hvo5,juliaastro_astrolib.jl,https://github.com/JuliaAstro/AstroLib.jl
1,hvo5,cra-ros-pkg_robot_localization,https://github.com/cra-ros-pkg/robot_localization
2,hvo5,hmri-group_hmri-toolbox,https://github.com/hMRI-group/hMRI-toolbox
3,hvo5,tdda_tdda,https://github.com/tdda/tdda
4,hvo5,openspace_openspace,https://github.com/OpenSpace/OpenSpace
5,hvo5,photon-hdf5_phconvert,https://github.com/Photon-HDF5/phconvert
6,hvo5,nocions_letswave6,https://github.com/NOCIONS/letswave6
7,hvo5,flatsurf_flatsurf,https://github.com/flatsurf/flatsurf
8,hvo5,mano-b_microsec,https://github.com/MANO-B/MicroSEC
9,hvo5,ablab_spades,https://github.com/ablab/spades


## Commit SHA1s from World of Code

`p2c` maps a WoC project id to commit SHA1s. If an id fails, the notebook tries the id derived from the GitHub URL (`owner_repo`, lowercased), which is the conversion used in the example notebook.


In [4]:
def reset_client():
    global woc
    woc = WocMapsRemote(base_url=WOC_URL)
    return woc


def github_owner_repo(url):
    parts = url.rstrip("/").split("/")
    return parts[-2], parts[-1]


def alternate_woc_id(url):
    owner, repo = github_owner_repo(url)
    return f"{owner}/{repo}".lower().replace("/", "_", 2)


def call_with_retry(label, fn, attempts=8):
    delay = 5
    last = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except KeyError:
            raise
        except Exception as exc:
            last = exc
            print(f"{label}: attempt {attempt} failed ({type(exc).__name__}: {exc})")
            time.sleep(delay)
            delay = min(delay * 2, 120)
            if attempt % 3 == 0:
                reset_client()
    raise RuntimeError(f"{label} failed after {attempts} attempts: {last}")


def project_shas(woc_id):
    values = call_with_retry(woc_id, lambda: woc.get_values("p2c", woc_id))
    if values is None:
        return []
    return list(values)


sha_path = Path("df_commits.csv")
if sha_path.exists() and sha_path.stat().st_size > 0:
    df = pd.read_csv(sha_path)
    print(f"Loaded cached SHA1 list from {sha_path} ({len(df)} rows)")
else:
    rows = []
    lookup_notes = []
    for rec in assigned.itertuples(index=False):
        prj = rec.WoC
        try:
            commits = project_shas(prj)
        except KeyError as exc:
            alt = alternate_woc_id(rec.GH)
            print(f"{prj}: not found ({exc}). Trying {alt}")
            commits = project_shas(alt)
            lookup_notes.append(f"{rec.WoC} -> {alt}")
            prj = alt
        print(f"{prj}: {len(commits)} commits")
        if len(commits) == 0:
            print(f"WARNING: {prj} returned 0 commits")
        for sha in commits:
            rows.append({"project": prj, "sha1": sha})
    df = pd.DataFrame(rows).drop_duplicates()
    df.to_csv(sha_path, index=False)
    print("lookup changes:", lookup_notes or "none")

print("SHA1 counts")
df.groupby("project").size().rename("commits")


juliaastro_astrolib.jl: 660 commits
cra-ros-pkg_robot_localization: 2428 commits
hmri-group_hmri-toolbox: 1613 commits


tdda_tdda: 1121 commits


openspace_openspace: 25521 commits
photon-hdf5_phconvert: 751 commits
nocions_letswave6: 311 commits
flatsurf_flatsurf: 2080 commits
mano-b_microsec: 654 commits


ablab_spades: 18459 commits
lookup changes: none
SHA1 counts


project
ablab_spades                      18459
cra-ros-pkg_robot_localization     2428
flatsurf_flatsurf                  2080
hmri-group_hmri-toolbox            1613
juliaastro_astrolib.jl              660
mano-b_microsec                     654
nocions_letswave6                   311
openspace_openspace               25521
photon-hdf5_phconvert               751
tdda_tdda                          1121
Name: commits, dtype: int64

## Commit content

Each commit SHA1 is looked up in `commit.tch` in chunks of 10, with a 1 second wait between chunks. Failed chunks are retried with backoff. Rows already stored in `df_commit_data.csv` are skipped, so this cell can be rerun after an interruption.


In [5]:
COMMIT_FIELDS = [
    "project",
    "commit",
    "tree",
    "parent",
    "author",
    "author_time",
    "author_tz",
    "committer",
    "committer_time",
    "committer_tz",
    "message",
]


def unwrap_commit(value):
    if isinstance(value, list) and value and isinstance(value[0], (list, tuple)):
        return value[0]
    return value


def parse_record(sha, record):
    record = unwrap_commit(record)
    if not isinstance(record, (list, tuple)) or len(record) < 5:
        raise ValueError(f"unexpected commit shape for {sha}: {str(record)[:180]}")
    parents = record[1]
    if isinstance(parents, (list, tuple)):
        parent = " ".join(str(p) for p in parents)
    else:
        parent = "" if parents is None else str(parents)
    author, committer = record[2], record[3]
    message = record[4] if record[4] is not None else ""
    return {
        "commit": sha,
        "tree": record[0],
        "parent": parent,
        "author": author[0],
        "author_time": int(author[1]),
        "author_tz": author[2],
        "committer": committer[0],
        "committer_time": int(committer[1]),
        "committer_tz": committer[2],
        "message": message,
    }


def fetch_chunk(shas):
    delay = 5
    last = None
    for attempt in range(1, 13):
        try:
            res, err = woc.get_values_many("commit.tch", list(shas))
            res = {k: unwrap_commit(v) for k, v in (res or {}).items()}
            return res, err or {}
        except Exception as exc:
            last = exc
            print(f"chunk retry {attempt} ({type(exc).__name__}: {exc})")
            time.sleep(delay)
            delay = min(delay * 2, 180)
            if attempt % 3 == 0:
                reset_client()
    raise RuntimeError(f"chunk failed after retries: {last}")


checkpoint = Path("df_commit_data.csv")
done = set()
if checkpoint.exists() and checkpoint.stat().st_size > 0:
    previous = pd.read_csv(checkpoint, usecols=["project", "commit"])
    done = set(zip(previous["project"], previous["commit"]))
    print(f"Resuming: {len(done)} commits already in {checkpoint.name}")

pending = [
    (project, sha)
    for project, sha in zip(df["project"], df["sha1"])
    if (project, sha) not in done
]
print(f"{len(pending)} commits still to fetch, {df['project'].nunique()} projects, {len(df)} SHA rows")

write_header = not checkpoint.exists() or checkpoint.stat().st_size == 0
out = checkpoint.open("a", newline="", encoding="utf-8")
writer = csv.DictWriter(out, fieldnames=COMMIT_FIELDS, extrasaction="ignore")
if write_header:
    writer.writeheader()
    out.flush()

failed = []
saved_now = 0
started = time.time()


def flush_rows(rows):
    global saved_now
    if not rows:
        return
    writer.writerows(rows)
    out.flush()
    os.fsync(out.fileno())
    saved_now += len(rows)


buffer = []
try:
    for start in range(0, len(pending), CHUNK):
        chunk_pairs = pending[start:start + CHUNK]
        projects_for = {}
        for project, sha in chunk_pairs:
            projects_for.setdefault(sha, []).append(project)
        res, err = fetch_chunk(list(projects_for))
        if err:
            print(f"API errors for {len(err)} sha1s in this chunk: {list(err)[:3]}")
        for sha, projects in projects_for.items():
            if sha not in res or res[sha] is None:
                for project in projects:
                    failed.append((project, sha, str((err or {}).get(sha, "missing from response"))))
                continue
            try:
                parsed = parse_record(sha, res[sha])
            except Exception as exc:
                for project in projects:
                    failed.append((project, sha, str(exc)))
                continue
            for project in projects:
                buffer.append({"project": project, **parsed})
        if len(buffer) >= 200:
            flush_rows(buffer)
            buffer = []
            done_total = len(done) + saved_now
            elapsed = time.time() - started
            print(f"saved {done_total}/{len(df)} commits ({elapsed/60:.1f} min)")
        time.sleep(SLEEP_SECONDS)
    flush_rows(buffer)
    buffer = []
finally:
    out.close()

# Second pass for commits the API did not return.
if failed:
    print(f"retrying {len(failed)} commits that failed the first pass")
    still_failed = []
    out = checkpoint.open("a", newline="", encoding="utf-8")
    writer = csv.DictWriter(out, fieldnames=COMMIT_FIELDS, extrasaction="ignore")
    try:
        for start in range(0, len(failed), CHUNK):
            group = failed[start:start + CHUNK]
            projects_for = {}
            for project, sha, _reason in group:
                projects_for.setdefault(sha, []).append(project)
            res, err = fetch_chunk(list(projects_for))
            rows = []
            for sha, projects in projects_for.items():
                if sha not in res or res[sha] is None:
                    reason = str((err or {}).get(sha, "missing from response"))
                    for project in projects:
                        still_failed.append((project, sha, reason))
                    continue
                try:
                    parsed = parse_record(sha, res[sha])
                except Exception as exc:
                    for project in projects:
                        still_failed.append((project, sha, str(exc)))
                    continue
                for project in projects:
                    rows.append({"project": project, **parsed})
            flush_rows(rows)
            time.sleep(SLEEP_SECONDS)
    finally:
        out.close()
    failed = still_failed

print(f"fetch finished in {(time.time()-started)/60:.1f} min; unresolved commits: {len(failed)}")
if failed:
    print("unresolved sample:", failed[:5])
if len(failed) > max(20, int(0.01 * len(df))):
    raise RuntimeError(f"{len(failed)} commits are still missing; rerun this cell to resume")


53598 commits still to fetch, 10 projects, 53598 SHA rows


API errors for 1 sha1s in this chunk: ['2369e4df5b846622dd0a68e9ceffea658a97a9e6']


saved 209/53598 commits (0.3 min)


API errors for 1 sha1s in this chunk: ['5d674b56e40febecc4f47aacf3e84f27b5122dff']


API errors for 2 sha1s in this chunk: ['7834eb815c8205bf495530c0bf0cee627c2e7936', '7895504120d07d30488a21e033a11df812fdb271']


...

saved 52987/53598 commits (92.9 min)


saved 53187/53598 commits (93.2 min)


saved 53387/53598 commits (93.5 min)


retrying 83 commits that failed the first pass


fetch finished in 94.0 min; unresolved commits: 83
unresolved sample: [('juliaastro_astrolib.jl', '2369e4df5b846622dd0a68e9ceffea658a97a9e6', 'Key 2369e4df5b846622dd0a68e9ceffea658a97a9e6 not found in /da5_fast/All.sha1c/commit_35.tch'), ('juliaastro_astrolib.jl', '5d674b56e40febecc4f47aacf3e84f27b5122dff', 'Key 5d674b56e40febecc4f47aacf3e84f27b5122dff not found in /da5_fast/All.sha1c/commit_93.tch'), ('juliaastro_astrolib.jl', '7834eb815c8205bf495530c0bf0cee627c2e7936', 'Key 7834eb815c8205bf495530c0bf0cee627c2e7936 not found in /da5_fast/All.sha1c/commit_120.tch'), ('juliaastro_astrolib.jl', '7895504120d07d30488a21e033a11df812fdb271', 'Key 7895504120d07d30488a21e033a11df812fdb271 not found in /da5_fast/All.sha1c/commit_120.tch'), ('juliaastro_astrolib.jl', '864550f8f4fe20a52d9c1f486c66ad0fd267484c', 'Key 864550f8f4fe20a52d9c1f486c66ad0fd267484c not found in /da5_fast/All.sha1c/commit_6.tch')]


In [6]:
# One retrieved commit, so the saved output shows the WoC record shape.
sample = pd.read_csv("df_commit_data.csv", nrows=1)
sample.T


,0
project,juliaastro_astrolib.jl
commit,00430dee0d6de3e87fe945e90a84775a82c519a9
tree,42d155a84adb0d28bcca706622cf8d4f7921815b
parent,b2a83edd44cb6f15343165af27ef31c6bee6cfc9
author,Documenter.jl <documenter@juliadocs.github.io>
author_time,1622754964
author_tz,0
committer,Documenter.jl <documenter@juliadocs.github.io>
committer_time,1622754964
committer_tz,0


## `hvo5_project_summary.csv`

Semicolon-separated, one row per commit. `time` is the author timestamp in Unix seconds. The file is rewritten from the cache (not appended), so running this cell again does not duplicate rows.


In [7]:
full = pd.read_csv("df_commit_data.csv")
full = full.drop_duplicates(subset=["project", "commit"])
summary = full[["project", "commit", "author", "author_time", "message"]].copy()
summary["author_time"] = pd.to_numeric(summary["author_time"], errors="coerce")
missing_time = int(summary["author_time"].isna().sum())
if missing_time:
    print(f"WARNING: dropping {missing_time} rows with no author time")
    summary = summary.dropna(subset=["author_time"])
summary["author_time"] = summary["author_time"].astype("int64")
summary["message"] = summary["message"].fillna("")
summary = summary.rename(columns={
    "project": "project_wocid",
    "commit": "commit_sha1",
    "author_time": "time",
    "message": "commit message",
})
summary = summary.sort_values(["project_wocid", "time", "commit_sha1"]).reset_index(drop=True)
summary_path = f"{NETID}_project_summary.csv"
summary.to_csv(summary_path, sep=";", index=False)
print(f"wrote {summary_path}: {len(summary)} rows, {summary['project_wocid'].nunique()} projects")

# Round-trip check: the semicolon file must parse back to the same number of commits.
check = pd.read_csv(summary_path, sep=";")
assert list(check.columns) == ["project_wocid", "commit_sha1", "author", "time", "commit message"]
assert len(check) == len(summary)
print("columns:", list(check.columns))
check.groupby("project_wocid").size().rename("rows")


wrote hvo5_project_summary.csv: 53515 rows, 10 projects
columns: ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']


project_wocid
ablab_spades                      18459
cra-ros-pkg_robot_localization     2414
flatsurf_flatsurf                  2079
hmri-group_hmri-toolbox            1609
juliaastro_astrolib.jl              651
mano-b_microsec                     654
nocions_letswave6                   311
openspace_openspace               25493
photon-hdf5_phconvert               751
tdda_tdda                          1094
Name: rows, dtype: int64

## World of Code summary

For each project: number of commits, number of distinct author strings, and the earliest and latest author timestamps. The same person with two emails is counted twice, because WoC identifies authors by the name and email stored on the commit.


In [8]:
woc_stats = summary.groupby("project_wocid").agg(
    commits=("commit_sha1", "nunique"),
    authors=("author", "nunique"),
    min_time=("time", "min"),
    max_time=("time", "max"),
).reset_index()
woc_stats["min_time_utc"] = pd.to_datetime(woc_stats["min_time"], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M:%S")
woc_stats["max_time_utc"] = pd.to_datetime(woc_stats["max_time"], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M:%S")
woc_stats


,project_wocid,commits,authors,min_time,max_time,min_time_utc,max_time_utc
0,ablab_spades,18459,196,1295899946,1762802524,2011-01-24 20:12:26,2025-11-10 19:22:04
1,cra-ros-pkg_robot_localization,2414,240,1396281291,1759317757,2014-03-31 15:54:51,2025-10-01 11:22:37
2,flatsurf_flatsurf,2079,10,1547297444,1774299346,2019-01-12 12:50:44,2026-03-23 20:55:46
3,hmri-group_hmri-toolbox,1609,42,1413813574,1761738579,2014-10-20 13:59:34,2025-10-29 11:49:39
4,juliaastro_astrolib.jl,651,39,1457563164,1774899097,2016-03-09 22:39:24,2026-03-30 19:31:37
5,mano-b_microsec,654,8,1604123752,1760652530,2020-10-31 05:55:52,2025-10-16 22:08:50
6,nocions_letswave6,311,25,1427529750,1752135472,2015-03-28 08:02:30,2025-07-10 08:17:52
7,openspace_openspace,25493,254,1388750541,1776467917,2014-01-03 12:02:21,2026-04-17 23:18:37
8,photon-hdf5_phconvert,751,15,1422059137,1771428974,2015-01-24 00:25:37,2026-02-18 15:36:14
9,tdda_tdda,1094,16,1462464058,1757088306,2016-05-05 16:00:58,2025-09-05 16:05:06


## GitHub summary

Stars and forks come from the repository endpoint. The last commit date is the committer date of the latest commit on the default branch. The default-branch commit count and the contributor count are only for the validation check below. WoC is not expected to match them exactly.


In [9]:
def github_headers():
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "cs445-mp2-hvo5",
    }
    try:
        token = subprocess.check_output(
            ["gh", "auth", "token"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        token = ""
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers


def last_page_count(response):
    """When per_page=1, the last page number is the item count."""
    link = response.headers.get("Link", "")
    match = re.search(r"[?&]page=(\d+)[^>]*>;\s*rel=\"last\"", link)
    if match:
        return int(match.group(1))
    if response.status_code == 200:
        body = response.json()
        if isinstance(body, list):
            return len(body)
    return None


session = requests.Session()
session.headers.update(github_headers())

gh_rows = []
for rec in assigned.itertuples(index=False):
    owner, repo = github_owner_repo(rec.GH)
    full_name = f"{owner}/{repo}"
    info = session.get(f"https://api.github.com/repos/{full_name}", timeout=60)
    info.raise_for_status()
    info_json = info.json()
    commits = session.get(
        f"https://api.github.com/repos/{full_name}/commits",
        params={"per_page": 1},
        timeout=60,
    )
    commits.raise_for_status()
    commit_json = commits.json()
    last_date = commit_json[0]["commit"]["committer"]["date"][:10] if commit_json else ""
    contributors = None
    for _ in range(3):
        contrib = session.get(
            f"https://api.github.com/repos/{full_name}/contributors",
            params={"per_page": 1, "anon": "true"},
            timeout=60,
        )
        if contrib.status_code == 202:
            time.sleep(3)
            continue
        contrib.raise_for_status()
        contributors = last_page_count(contrib)
        break
    gh_rows.append({
        "project_wocid": rec.WoC,
        "github": full_name,
        "stars": int(info_json["stargazers_count"]),
        "forks": int(info_json["forks_count"]),
        "last_commit_date": last_date,
        "gh_default_branch_commits": last_page_count(commits),
        "gh_contributors": contributors,
    })
    time.sleep(0.3)

gh_stats = pd.DataFrame(gh_rows)
gh_stats


,project_wocid,github,stars,forks,last_commit_date,gh_default_branch_commits,gh_contributors
0,juliaastro_astrolib.jl,JuliaAstro/AstroLib.jl,91,25,2026-07-08,317,24
1,cra-ros-pkg_robot_localization,cra-ros-pkg/robot_localization,1962,1039,2026-07-16,1074,81
2,hmri-group_hmri-toolbox,hMRI-group/hMRI-toolbox,86,46,2026-09-16,1463,24
3,tdda_tdda,tdda/tdda,311,29,2026-07-13,1419,12
4,openspace_openspace,OpenSpace/OpenSpace,1259,210,2026-09-26,12638,69
5,photon-hdf5_phconvert,Photon-HDF5/phconvert,17,14,2026-09-08,716,11
6,nocions_letswave6,NOCIONS/letswave6,50,32,2026-09-20,314,23
7,flatsurf_flatsurf,flatsurf/flatsurf,10,4,2025-12-17,1539,5
8,mano-b_microsec,MANO-B/MicroSEC,6,2,2025-10-16,653,2
9,ablab_spades,ablab/spades,961,165,2026-09-28,15354,140


## Validation

GitHub's commit count is the default branch only. WoC counts every commit it has stored for that project id, including other branches, and it stops around January 2025. Author counts also differ: WoC counts distinct `name <email>` strings, while GitHub's contributor list collapses identities and can omit some commits.


In [10]:
comparison = woc_stats.merge(gh_stats, on="project_wocid", how="outer")
comparison["commit_difference_woc_minus_github"] = (
    comparison["commits"] - comparison["gh_default_branch_commits"]
)
comparison["author_difference_woc_minus_github"] = (
    comparison["authors"] - comparison["gh_contributors"]
)
cols = [
    "project_wocid",
    "commits",
    "gh_default_branch_commits",
    "commit_difference_woc_minus_github",
    "authors",
    "gh_contributors",
    "author_difference_woc_minus_github",
    "max_time_utc",
    "last_commit_date",
]
comparison[cols]


,project_wocid,commits,gh_default_branch_commits,commit_difference_woc_minus_github,authors,gh_contributors,author_difference_woc_minus_github,max_time_utc,last_commit_date
0,ablab_spades,18459,15354,3105,196,140,56,2025-11-10 19:22:04,2026-09-28
1,cra-ros-pkg_robot_localization,2414,1074,1340,240,81,159,2025-10-01 11:22:37,2026-07-16
2,flatsurf_flatsurf,2079,1539,540,10,5,5,2026-03-23 20:55:46,2025-12-17
3,hmri-group_hmri-toolbox,1609,1463,146,42,24,18,2025-10-29 11:49:39,2026-09-16
4,juliaastro_astrolib.jl,651,317,334,39,24,15,2026-03-30 19:31:37,2026-07-08
5,mano-b_microsec,654,653,1,8,2,6,2025-10-16 22:08:50,2025-10-16
6,nocions_letswave6,311,314,-3,25,23,2,2025-07-10 08:17:52,2026-09-20
7,openspace_openspace,25493,12638,12855,254,69,185,2026-04-17 23:18:37,2026-09-26
8,photon-hdf5_phconvert,751,716,35,15,11,4,2026-02-18 15:36:14,2026-09-08
9,tdda_tdda,1094,1419,-325,16,12,4,2025-09-05 16:05:06,2026-07-13


## GitHub repository information

For each assigned project: number of stars, number of forks, and last commit date on the default branch (UTC), retrieved on 2026-09-28.

| Project (WoC id) | GitHub repository | Stars | Forks | Last commit date |
|---|---|---:|---:|---|
| juliaastro_astrolib.jl | JuliaAstro/AstroLib.jl | 91 | 25 | 2026-07-08 |
| cra-ros-pkg_robot_localization | cra-ros-pkg/robot_localization | 1962 | 1039 | 2026-07-16 |
| hmri-group_hmri-toolbox | hMRI-group/hMRI-toolbox | 86 | 46 | 2026-09-16 |
| tdda_tdda | tdda/tdda | 311 | 29 | 2026-07-13 |
| openspace_openspace | OpenSpace/OpenSpace | 1259 | 210 | 2026-09-26 |
| photon-hdf5_phconvert | Photon-HDF5/phconvert | 17 | 14 | 2026-09-08 |
| nocions_letswave6 | NOCIONS/letswave6 | 50 | 32 | 2026-09-20 |
| flatsurf_flatsurf | flatsurf/flatsurf | 10 | 4 | 2025-12-17 |
| mano-b_microsec | MANO-B/MicroSEC | 6 | 2 | 2025-10-16 |
| ablab_spades | ablab/spades | 961 | 165 | 2026-09-28 |


## World of Code information

For each assigned project: number of commits, number of authors (distinct `name <email>` strings), and minimum and maximum author time (Unix seconds, with the UTC date).

| Project (WoC id) | Commits | Authors | Min time | Max time |
|---|---:|---:|---|---|
| juliaastro_astrolib.jl | 651 | 39 | 1457563164 (2016-03-09) | 1774899097 (2026-03-30) |
| cra-ros-pkg_robot_localization | 2414 | 240 | 1396281291 (2014-03-31) | 1759317757 (2025-10-01) |
| hmri-group_hmri-toolbox | 1609 | 42 | 1413813574 (2014-10-20) | 1761738579 (2025-10-29) |
| tdda_tdda | 1094 | 16 | 1462464058 (2016-05-05) | 1757088306 (2025-09-05) |
| openspace_openspace | 25493 | 254 | 1388750541 (2014-01-03) | 1776467917 (2026-04-17) |
| photon-hdf5_phconvert | 751 | 15 | 1422059137 (2015-01-24) | 1771428974 (2026-02-18) |
| nocions_letswave6 | 311 | 25 | 1427529750 (2015-03-28) | 1752135472 (2025-07-10) |
| flatsurf_flatsurf | 2079 | 10 | 1547297444 (2019-01-12) | 1774299346 (2026-03-23) |
| mano-b_microsec | 654 | 8 | 1604123752 (2020-10-31) | 1760652530 (2025-10-16) |
| ablab_spades | 18459 | 196 | 1295899946 (2011-01-24) | 1762802524 (2025-11-10) |

WoC listed 53,598 commit SHA1s for these projects (`p2c`). Content was retrieved for 53,515. The other 83 (robot_localization 14, tdda 27, openspace 28, astrolib.jl 9, hmri-toolbox 4, flatsurf 1) are listed in `p2c` but missing from WoC's commit store: both `commit.tch` and `show_content('commit', ...)` return "Key not found". They are excluded from the counts above and from `hvo5_project_summary.csv`.


## Validation notes

- **Commit counts.** WoC counts are close to GitHub's default-branch counts for mano-b_microsec (654 vs 653), nocions_letswave6 (311 vs 314), photon-hdf5_phconvert (751 vs 716), and hmri-group_hmri-toolbox (1609 vs 1463). WoC is much higher for openspace_openspace (25,493 vs 12,638), cra-ros-pkg_robot_localization (2,414 vs 1,074), ablab_spades (18,459 vs 15,354), and juliaastro_astrolib.jl (651 vs 317). WoC keeps commits from every branch it has seen (robot_localization has a separate branch per ROS release, and astrolib.jl includes its `gh-pages` documentation builds), while GitHub's count covers the default branch only. tdda_tdda is the one project where WoC is lower (1,094 vs 1,419). About 325 commits are missing, 27 of them the unresolved SHA1s above, and the rest are commits made after WoC's last crawl of the project (WoC's latest tdda commit is 2025-09-05, and GitHub's is 2026-07-13).
- **Author counts.** WoC author counts are higher than GitHub contributor counts for every project, because WoC counts each distinct `name <email>` string, so one person using several emails or machines is counted more than once. GitHub merges identities by account and leaves out commits it cannot link to one.
- **Time range.** The version of WoC served by this API goes past the January 2025 cutoff mentioned in the assignment. Max times run from 2025-07 to 2026-04. Every project has GitHub commits after WoC's max time except mano-b_microsec, where the two dates are the same (2025-10-16), and flatsurf_flatsurf, where WoC's latest commit (2026-03-23) is newer than GitHub's default-branch date (2025-12-17), so it came from another branch.
